# 07. L4 の上限を探る

このノートは、[colab-oss-lab](https://github.com/moruku36/colab-oss-lab) の実験 07 です。

[実験02](../docs/results/02_quantization_max.md) では Gemma 4 31B（4bit）が VRAM 17.0GB で載り、約 5GB 余りました。
では **どこまで大きいモデルが載るのか**、そして **載ったとき、どれだけ長い文章を扱えるのか** を確かめます。

エンジンと量子化は 02 / 05 と同じ **transformers + bitsandbytes 4bit NF4**、モデルは **GPU だけに載せる**（CPU に逃がさない）。

| 構成 | モデル | 大きさ | bf16 のファイル | 4bit の見積もり |
|---|---|---|---|---|
| 参考（02） | Gemma 4 31B-it | 31.3B（密） | 62.5GB | 実測 17.0GB |
| A | [Qwen/Qwen3-32B](https://huggingface.co/Qwen/Qwen3-32B) | 32.8B（密） | 65.5GB | 約 19GB |
| B | [ByteDance-Seed/Seed-OSS-36B-Instruct](https://huggingface.co/ByteDance-Seed/Seed-OSS-36B-Instruct) | 36.2B（密） | 72.3GB | 約 21GB（ぎりぎり） |

どちらも Apache-2.0、ゲートなし。

測るもの（載った場合）:

1. 読み込み後の VRAM と、残りの空き
2. **扱える文章の長さ（コンテキスト）の上限** … 1K → 2K → 4K → 8K → 16K → 32K トークンと伸ばして、メモリ不足になる直前まで
3. 速さ（1件ずつ 128 トークン）
4. 05 と同じ 10 問の正答数、日本語の返事

「想定どおり」とは:

- 32B（A）は L4 に載る
- 36B（B）も載るが、空きは 2GB 以下（ここが上限付近）
- 大きいモデルほど、扱える文章の長さの上限が短くなる
- 載ったモデルは 10 問中 7 問以上正解する（4bit でも賢さを保つ）

所要時間の目安: 40〜60 分（ダウンロードが合計 138GB）。

---

## 実行する前に

1. **ランタイム → ランタイムのタイプを変更 → L4 GPU → Save**（High-RAM）
2. 上から順に ▶（または「すべてのセルを実行」）
3. 終わったら **ランタイム → セッションを管理 → 解放**

## 1. GPU とディスクを確認して、ライブラリを入れる

In [ ]:
# ノート本体では torch を使わない（各構成は別プロセスで動かし、GPU メモリを毎回まっさらにするため）
import shutil, subprocess
q = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], text=True)
gpu_name, mem_mib = [x.strip() for x in q.strip().split(",")]
vram_total_gb = int(mem_mib) / 1024
disk_free_gb = shutil.disk_usage("/").free / 1024**3
assert "L4" in gpu_name, f"GPU が L4 ではありません: {gpu_name}"
print("GPU:", gpu_name, round(vram_total_gb, 1), "GB / ディスク空き:", round(disk_free_gb), "GB（ダウンロードに約 138GB 使う）")
assert disk_free_gb > 150, "ディスクが足りません"

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes 2>&1 | tail -2
!python -c "import torch, transformers, bitsandbytes; print('torch', torch.__version__, '/ transformers', transformers.__version__, '/ bitsandbytes', bitsandbytes.__version__)"

## 2. 計測用のスクリプトを書く

- 読み込みは `device_map={"": 0}` で **GPU 0 だけ**に載せる。載らなければメモリ不足のエラーになる（＝上限を超えた）
- 文章の長さの上限は、ダミーの文章を N トークン入れて 4 トークン生成できるかで調べる。メモリ不足になったら、その手前が上限

In [ ]:
%%writefile limit.py
import argparse, json, math, re, time, traceback
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

p = argparse.ArgumentParser()
p.add_argument("--name", required=True)
p.add_argument("--model", required=True)
a = p.parse_args()
res = dict(name=a.name, model=a.model, loaded=False)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

def save():
    json.dump(res, open(f"/content/limit_{a.name}.json", "w"), ensure_ascii=False, indent=1)

tok = AutoTokenizer.from_pretrained(a.model)
tok.padding_side = "left"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
t0 = time.time()
try:
    model = AutoModelForCausalLM.from_pretrained(a.model, quantization_config=bnb, dtype=torch.bfloat16, device_map={"": 0})
except Exception as e:  # bitsandbytes は RuntimeError（CUDA out of memory）で落ちることもある
    if not (isinstance(e, torch.OutOfMemoryError) or "out of memory" in str(e).lower()):
        raise
    res.update(error="読み込み中にメモリ不足: " + str(e).split("\n")[0][:300],
               vram_at_fail_gb=torch.cuda.memory_allocated() / 1024**3, load_sec=time.time() - t0)
    save(); print("FAIL", res["error"]); raise SystemExit(0)
model.eval()
torch.cuda.empty_cache()
res.update(loaded=True, load_sec=time.time() - t0,
           vram_load_gb=torch.cuda.memory_allocated() / 1024**3,
           free_gb=torch.cuda.mem_get_info()[0] / 1024**3,
           n_params=sum(p.numel() for p in model.parameters()))
print(f"読み込み OK: VRAM {res['vram_load_gb']:.2f}GB / 空き {res['free_gb']:.2f}GB", flush=True)
save()

CT = dict(enable_thinking=False, thinking_budget=0)  # Qwen3 と Seed-OSS の両方で「考えずにすぐ答える」
def chat(q, system=None):
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": q}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, **CT)
def clean(t):
    t = re.sub(r"<think>.*?</think>", "", t, flags=re.S)
    t = re.sub(r"<seed:think>.*?</seed:think>", "", t, flags=re.S)
    return t.strip()

# 1) 扱える文章の長さの上限
filler = open("/content/filler.txt").read()
filler_ids = tok(filler, add_special_tokens=False).input_ids
ctx = []
for n in [1024, 2048, 4096, 8192, 16384, 32768]:
    ids = (filler_ids * (n // len(filler_ids) + 1))[:n]
    x = torch.tensor([ids], device=0)
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    try:
        with torch.no_grad():
            t = time.time()
            model.generate(input_ids=x, attention_mask=torch.ones_like(x), max_new_tokens=4, do_sample=False)
            sec = time.time() - t
        ctx.append(dict(n=n, ok=True, sec=sec, peak_gb=torch.cuda.max_memory_allocated() / 1024**3))
        print(f"  {n:>6} トークン: OK（{sec:.1f} 秒、ピーク {ctx[-1]['peak_gb']:.2f}GB）", flush=True)
    except torch.OutOfMemoryError:
        ctx.append(dict(n=n, ok=False))
        print(f"  {n:>6} トークン: メモリ不足", flush=True)
        del x; torch.cuda.empty_cache()
        break
    del x
res["context"] = ctx
res["max_context"] = max([c["n"] for c in ctx if c["ok"]], default=0)
save()
torch.cuda.empty_cache()

# 2) 速さ（1件ずつ、ちょうど 128 トークン）
enc = tok([chat("小学校の児童にも分かる言葉で、GPUとVRAMの違いを3文で説明してください。")], return_tensors="pt").to(0)
with torch.no_grad():
    model.generate(**enc, max_new_tokens=8, do_sample=False)
    torch.cuda.synchronize(); t = time.time()
    model.generate(**enc, max_new_tokens=128, min_new_tokens=128, do_sample=False)
    torch.cuda.synchronize(); res["tps"] = 128 / (time.time() - t)
    out = model.generate(**enc, max_new_tokens=200, do_sample=False)
res["sample"] = clean(tok.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True))
save()

# 3) 10問（05 と同じ）
SYSTEM = "You are a helpful assistant. Answer in Japanese. 最後の行に必ず「答え: <数字>」の形で答えだけを書いてください。"
QUIZ = [
    ("ある数に3を足して2倍すると、その数の3倍より4小さくなります。ある数はいくつですか。", 10),
    ("1から100までの整数のうち、3でも5でも割り切れないものはいくつありますか。", 53),
    ("英単語 strawberry の中に、アルファベットの r は何個含まれていますか。", 3),
    ("A、B、C、D の4人が横一列に並びます。AとBが隣り合わない並び方は何通りですか。", 12),
    ("時計が3時15分を指しているとき、長針と短針がつくる小さいほうの角は何度ですか。", 7.5),
    ("7で割ると3余り、5で割ると2余る2桁の自然数のうち、いちばん小さいものは何ですか。", 17),
    ("定価の2割引きで買った品物の代金が960円でした。定価は何円ですか。", 1200),
    ("1から50までの整数をすべて足すといくつですか。", 1275),
    ("サイコロを2個振ったとき、出た目の和が7になる出方は、36通りのうち何通りですか。", 6),
    ("A地点からB地点まで、時速4kmで歩くと時速12kmの自転車より1時間遅く着きます。AB間の距離は何kmですか。", 6),
]
def extract(text):
    text = text.replace(",", "")
    m = re.findall(r"答え\s*[:：]\s*\**\s*([0-9]+(?:\.[0-9]+)?)", text) or re.findall(r"([0-9]+(?:\.[0-9]+)?)", text)
    return float(m[-1]) if m else None
answers = []
bs = 5
i = 0
while i < len(QUIZ):
    chunk = QUIZ[i:i + bs]
    try:
        enc = tok([chat(q, SYSTEM) for q, _ in chunk], return_tensors="pt", padding=True).to(0)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=384, do_sample=False)
        answers += [clean(tok.decode(o[enc.input_ids.shape[1]:], skip_special_tokens=True)) for o in out]
        i += len(chunk)
    except torch.OutOfMemoryError:
        torch.cuda.empty_cache()
        if bs == 1:
            raise
        bs = 1  # メモリが足りなければ 1 問ずつにする
res["quiz_batch"] = bs
res["quiz"] = [dict(q=q, expected=e, got=extract(t), correct=(g := extract(t)) is not None and abs(g - e) < 1e-6, text=t)
               for (q, e), t in zip(QUIZ, answers)]
res["quiz_correct"] = sum(r["correct"] for r in res["quiz"])
res["vram_peak_gb"] = torch.cuda.max_memory_allocated() / 1024**3
save()
print(f"OK {a.name}: 上限 {res['max_context']} トークン / {res['tps']:.1f} tok/s / 10問 {res['quiz_correct']}")

In [ ]:
# 文章の長さを測るためのダミーの文章（このリポジトリの解説ページ）
import urllib.request
base = "https://raw.githubusercontent.com/moruku36/colab-oss-lab/main/docs/"
pages = ["01-what-is-colab.md", "02-google-ai-pro.md", "03-what-you-can-do.md", "04-oss-models.md", "05-gpu-basics.md", "06-quantization.md"]
open("/content/filler.txt", "w").write("\n\n".join(urllib.request.urlopen(base + p).read().decode("utf-8") for p in pages))
print("OK")

## 3. 2つのモデルを順に計測する

In [ ]:
import json, os, re, subprocess, time

MODELS = [
    ("A_qwen3_32b", "Qwen3-32B（32.8B）", "Qwen/Qwen3-32B"),
    ("B_seed_oss_36b", "Seed-OSS-36B（36.2B）", "ByteDance-Seed/Seed-OSS-36B-Instruct"),
]
results = {}
for name, label, model_id in MODELS:
    if os.path.exists(f"/content/limit_{name}.json") and "quiz" in json.load(open(f"/content/limit_{name}.json")):
        results[name] = json.load(open(f"/content/limit_{name}.json")); results[name]["label"] = label
        print(f"[{label}] 前回の結果を使う"); continue
    t = time.time()
    p = subprocess.run(["python", "limit.py", "--name", name, "--model", model_id], capture_output=True, text=True)
    log = p.stdout + p.stderr
    open(f"/content/limit_{name}.log", "w").write(log)
    print(f"[{label}]（{(time.time() - t) / 60:.1f} 分）")
    for line in p.stdout.splitlines():
        print("  " + line)
    if os.path.exists(f"/content/limit_{name}.json"):
        results[name] = json.load(open(f"/content/limit_{name}.json")); results[name]["label"] = label
    if p.returncode != 0:
        causes = [l for l in log.splitlines() if re.search(r"(Error|error:|out of memory)", l)]
        print("  失敗:", causes[-1][:300] if causes else "ログを確認: /content/limit_" + name + ".log")
    # 次のモデルのためにディスクを空ける（1 モデル 65〜72GB）
    subprocess.run(f"rm -rf ~/.cache/huggingface/hub/models--{model_id.replace('/', '--')}", shell=True)

## 4. まとめて、実行記録を出す

In [ ]:
from datetime import datetime, timezone, timedelta
import torch, transformers, bitsandbytes  # バージョン表示のためだけ

A, B = results.get("A_qwen3_32b"), results.get("B_seed_oss_36b")
def loaded(r): return bool(r and r.get("loaded"))
checks = {
    "32B（A）は L4 に載る": loaded(A),
    "36B（B）も載るが、空きは 2GB 以下": loaded(B) and B["free_gb"] <= 2.0,
    "大きいモデルほど、扱える文章の長さの上限が短くなる": loaded(A) and loaded(B) and B.get("max_context", 0) < A.get("max_context", 0),
    "載ったモデルは 10 問中 7 問以上正解": all(r.get("quiz_correct", 0) >= 7 for r in (A, B) if loaded(r)) and (loaded(A) or loaded(B)),
}
ok = all(checks.values())
now = datetime.now(timezone(timedelta(hours=9))).strftime("%Y-%m-%d %H:%M JST")
L = ["# 実行記録: 07 L4 の上限を探る", "",
     f"- 実行日: {now}", "- 実行場所: Google Colab",
     f"- GPU: {gpu_name} / VRAM {round(vram_total_gb, 1)} GB",
     f"- torch {torch.__version__} / transformers {transformers.__version__} / bitsandbytes {bitsandbytes.__version__}",
     "- 量子化: bitsandbytes 4bit NF4（double quant）、GPU だけに載せる",
     f"- 想定どおりか: {'はい' if ok else 'いいえ'}", "",
     "## まとめ", "",
     "| 構成 | 載った？ | VRAM（読み込み後） | 空き | 文章の長さの上限 | 速さ（トークン/秒） | 10問 |",
     "|---|---|---|---|---|---|---|",
     "| 参考: Gemma 4 31B（実験02） | ○ | 17.0 GB | 約 5 GB | 測っていない | 約 5.5 | - |"]
for name, label, _ in MODELS:
    r = results.get(name)
    if not loaded(r):
        L.append(f"| {label} | × | - | - | - | - | - |")
        continue
    L.append(f"| {label} | ○ | {r['vram_load_gb']:.2f} GB | {r['free_gb']:.2f} GB | {r.get('max_context', 0):,} トークン |"
             f" {r.get('tps', 0):.1f} | {r.get('quiz_correct', '-')} / 10 |")
L += ["", "## 判定", ""] + [f"- [{'x' if v else ' '}] {k}" for k, v in checks.items()] + [""]
for name, label, _ in MODELS:
    r = results.get(name)
    L += [f"## {label}", ""]
    if not r:
        L += ["- 結果なし（ログを確認）", ""]; continue
    if not r.get("loaded"):
        L += [f"- 載らなかった: {r.get('error')}", f"- そのときの VRAM: {r.get('vram_at_fail_gb', 0):.2f} GB", ""]; continue
    L += [f"- 読み込み: {r['load_sec'] / 60:.1f} 分（ダウンロード込み）/ パラメータ（4bit は詰めて数える）: {r['n_params']:,}",
          "- 文章の長さ: " + " / ".join(f"{c['n']:,}: " + (f"OK {c['sec']:.1f}秒 ピーク {c['peak_gb']:.2f}GB" if c['ok'] else "メモリ不足") for c in r.get("context", [])),
          f"- VRAM ピーク（10問のとき）: {r.get('vram_peak_gb', 0):.2f} GB / 10問は {r.get('quiz_batch')} 問ずつ生成",
          "- 10問: " + " ".join(("○" if q["correct"] else "×") + str(q["got"]) for q in r.get("quiz", [])), "",
          "返事の例（GPUとVRAMの違いを3文で）:", "", "```", r.get("sample", "")[:600], "```", ""]
print("\n".join(L))

## 終わったら

**ランタイム → セッションを管理 → 解放** を必ず押してください。